# Ollama Portfolio Analyzer

This notebook keeps the portfolio math deterministic in Python and uses a local Ollama model for the narrative analysis.

It is designed to be easy to inspect and debug cell-by-cell while you iterate.


In [ ]:
# If you need packages in the notebook environment, run this once:
# %pip install pandas


In [2]:
from __future__ import annotations

import json
import math
import re
import subprocess
import urllib.error
import urllib.request
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd


In [3]:
MODEL_NAME = "llama3.3:latest"
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
CSV_PATH = Path("../data/raw/mantis_invest.csv")
RISK_PROFILE = 65
MONTHLY_BUDGET = 1500.0
SECTORS = "Technology, Healthcare, ETFs"
NUM_PREDICT = 900
TEMPERATURE = 0.2


In [4]:
@dataclass
class PositionLot:
    quantity: float
    unit_cost: float
    buy_date: datetime


def parse_money(value: Any) -> float:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return 0.0
    text = str(value).strip().replace("$", "").replace(",", "")
    if not text:
        return 0.0
    negative = text.startswith("(") and text.endswith(")")
    if negative:
        text = text[1:-1]
    try:
        amount = float(text)
    except ValueError:
        return 0.0
    return -amount if negative else amount


def parse_quantity(value: Any) -> float:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return 0.0
    text = str(value).strip().replace(",", "")
    if not text:
        return 0.0
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    return float(match.group(0)) if match else 0.0


def load_transactions(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    df["Activity Date"] = pd.to_datetime(df["Activity Date"], errors="coerce")
    df["Amount_num"] = df.get("Amount", 0).apply(parse_money)
    df["Price_num"] = df.get("Price", 0).apply(parse_money)
    df["Quantity_num"] = df.get("Quantity", 0).apply(parse_quantity)
    return df.sort_values("Activity Date").reset_index(drop=True)


def holding_period_bucket(days: float) -> str:
    if days <= 30:
        return "<=30d"
    if days < 365:
        return "31-364d"
    return ">=365d"


In [5]:
def summarize_portfolio(df: pd.DataFrame) -> dict[str, Any]:
    buy_df = df[df["Trans Code"] == "Buy"]
    sell_df = df[df["Trans Code"] == "Sell"]
    ach_df = df[df["Trans Code"] == "ACH"]
    dividend_df = df[df["Trans Code"] == "CDIV"]
    interest_df = df[df["Trans Code"] == "INT"]

    lots: dict[str, deque[PositionLot]] = defaultdict(deque)
    realized_pnl: dict[str, float] = defaultdict(float)
    holding_periods: list[float] = []

    for _, row in df.iterrows():
        symbol = str(row.get("Instrument") or "").strip()
        code = str(row.get("Trans Code") or "").strip()
        if not symbol:
            continue

        if code == "Buy":
            quantity = float(row["Quantity_num"])
            cash_out = -float(row["Amount_num"])
            if quantity > 0:
                lots[symbol].append(
                    PositionLot(
                        quantity=quantity,
                        unit_cost=cash_out / quantity,
                        buy_date=row["Activity Date"].to_pydatetime(),
                    )
                )
        elif code == "SPL":
            extra_qty = float(row["Quantity_num"])
            total_qty = sum(lot.quantity for lot in lots[symbol])
            if total_qty > 0 and extra_qty > 0:
                factor = (total_qty + extra_qty) / total_qty
                adjusted = deque()
                for lot in lots[symbol]:
                    adjusted.append(
                        PositionLot(
                            quantity=lot.quantity * factor,
                            unit_cost=lot.unit_cost / factor,
                            buy_date=lot.buy_date,
                        )
                    )
                lots[symbol] = adjusted
        elif code == "Sell":
            remaining = float(row["Quantity_num"])
            sell_date = row["Activity Date"].to_pydatetime()
            sell_amount = float(row["Amount_num"])
            sell_unit_price = sell_amount / remaining if remaining else 0.0

            while remaining > 1e-9 and lots[symbol]:
                lot = lots[symbol][0]
                take = min(remaining, lot.quantity)
                realized_pnl[symbol] += take * (sell_unit_price - lot.unit_cost)
                holding_periods.append((sell_date - lot.buy_date).days)
                lot.quantity -= take
                remaining -= take
                if lot.quantity <= 1e-9:
                    lots[symbol].popleft()

    current_positions = []
    for symbol, queue in lots.items():
        total_qty = sum(lot.quantity for lot in queue)
        if total_qty <= 1e-9:
            continue
        total_cost = sum(lot.quantity * lot.unit_cost for lot in queue)
        current_positions.append(
            {
                "ticker": symbol,
                "quantity": round(total_qty, 6),
                "cost_basis_total": round(total_cost, 2),
            }
        )

    current_positions.sort(key=lambda item: item["cost_basis_total"], reverse=True)
    total_open_cost = sum(item["cost_basis_total"] for item in current_positions)
    for item in current_positions:
        item["portfolio_cost_weight"] = (
            round((item["cost_basis_total"] / total_open_cost) * 100, 2)
            if total_open_cost
            else 0.0
        )

    buys_by_ticker = (
        buy_df.groupby("Instrument")["Amount_num"].sum().abs().sort_values(ascending=False)
    )
    sells_by_ticker = sell_df.groupby("Instrument")["Amount_num"].sum().sort_values(
        ascending=False
    )
    years = max(
        ((df["Activity Date"].max() - df["Activity Date"].min()).days / 365.25), 0.01
    )
    holding_counter = Counter(holding_period_bucket(days) for days in holding_periods)

    return {
        "date_range": {
            "start": df["Activity Date"].min().date().isoformat(),
            "end": df["Activity Date"].max().date().isoformat(),
            "years": round(years, 2),
        },
        "transaction_counts": {
            "rows": int(len(df)),
            "buys": int(len(buy_df)),
            "sells": int(len(sell_df)),
            "distinct_symbols_traded": int(
                df.loc[df["Instrument"].notna(), "Instrument"].nunique()
            ),
        },
        "cash_flows": {
            "net_deposits": round(float(ach_df["Amount_num"].sum()), 2),
            "gross_buys": round(float(buy_df["Amount_num"].sum() * -1), 2),
            "gross_sells": round(float(sell_df["Amount_num"].sum()), 2),
            "dividends": round(float(dividend_df["Amount_num"].sum()), 2),
            "interest": round(float(interest_df["Amount_num"].sum()), 2),
        },
        "behavioral_metrics": {
            "sell_to_buy_ratio": round(
                float(sell_df["Amount_num"].sum())
                / max(float(buy_df["Amount_num"].sum() * -1), 1.0),
                3,
            ),
            "annualized_turnover_proxy": round(
                (
                    float(sell_df["Amount_num"].sum())
                    / max(float(buy_df["Amount_num"].sum() * -1), 1.0)
                )
                / years,
                3,
            ),
            "holding_period_buckets": dict(holding_counter),
            "median_holding_period_days": (
                round(float(pd.Series(holding_periods).median()), 1)
                if holding_periods
                else None
            ),
        },
        "current_positions_top_15": current_positions[:15],
        "top_deployed_capital": [
            {
                "ticker": ticker,
                "gross_buy_amount": round(float(amount * -1), 2),
                "gross_sell_amount": round(float(sells_by_ticker.get(ticker, 0.0)), 2),
                "realized_pnl": round(float(realized_pnl.get(ticker, 0.0)), 2),
            }
            for ticker, amount in buys_by_ticker.head(15).items()
        ],
        "realized_pnl_by_ticker": [
            {"ticker": ticker, "realized_pnl": round(float(pnl), 2)}
            for ticker, pnl in sorted(
                realized_pnl.items(), key=lambda item: item[1], reverse=True
            )[:15]
        ],
    }


In [6]:
def ollama_tags(base_url: str = OLLAMA_BASE_URL) -> dict[str, Any]:
    request = urllib.request.Request(f"{base_url}/api/tags", method="GET")
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))


def ollama_available(model_name: str, base_url: str = OLLAMA_BASE_URL) -> bool:
    try:
        payload = ollama_tags(base_url)
    except Exception:
        return False
    names = {item.get("name") for item in payload.get("models", [])}
    return model_name in names


def pull_model(model_name: str) -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        ["ollama", "pull", model_name],
        check=True,
        text=True,
        capture_output=True,
    )


try:
    tags = ollama_tags()
    print("Ollama is reachable.")
    print("Available models:")
    print([item.get("name") for item in tags.get("models", [])])
except Exception as exc:
    print("Could not reach Ollama at", OLLAMA_BASE_URL)
    print("Start it in a terminal with: ollama serve")
    print("Error:", exc)


Ollama is reachable.
Available models:
['llama3.3:latest', 'llama3.1:latest', 'gemma:2b']


In [7]:
if not ollama_available(MODEL_NAME):
    print(f"Pulling {MODEL_NAME}...")
    result = pull_model(MODEL_NAME)
    print(result.stdout)
else:
    print(f"{MODEL_NAME} is already available locally.")


llama3.3:latest is already available locally.


In [8]:
def build_messages(
    portfolio_summary: dict[str, Any],
    risk_profile: int,
    monthly_budget: float,
    sectors: str,
) -> list[dict[str, str]]:
    system_prompt = (
        "You are an investment portfolio analysis assistant. "
        "Use only the structured portfolio data provided to you. "
        "Do not invent prices, news, or fundamentals. "
        "Do not claim certainty about future returns. "
        "Provide educational portfolio analysis rather than guaranteed financial advice. "
        "Return concise markdown with these sections: "
        "1. Investor Profile "
        "2. Portfolio Strengths "
        "3. Portfolio Risks "
        "4. Risk-Profile Mismatch "
        "5. Holdings to Review "
        "6. Long-Term Allocation Ideas "
        "7. Important Caveats"
    )

    user_prompt = (
        f"User input:\n"
        f"- Risk profile: {risk_profile}/100\n"
        f"- Monthly budget: ${monthly_budget:,.2f}\n"
        f"- Preferred sectors: {sectors}\n\n"
        "Structured portfolio summary:\n"
        f"{json.dumps(portfolio_summary, indent=2)}\n\n"
        "Analyze this investor and provide:\n"
        "- an observed risk appetite estimate with reasoning\n"
        "- whether the current portfolio fits the stated risk profile\n"
        "- the main concentration/diversification issues\n"
        "- which holdings deserve review based on concentration or behavior patterns\n"
        "- a cautious long-term monthly allocation approach based on the stated budget\n"
        "- explicit caveats about what cannot be concluded from transaction history alone"
    )

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


def call_ollama(
    model: str,
    messages: list[dict[str, str]],
    base_url: str = OLLAMA_BASE_URL,
    temperature: float = TEMPERATURE,
    num_predict: int = NUM_PREDICT,
) -> str:
    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": num_predict,
        },
    }
    data = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        f"{base_url}/api/chat",
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    try:
        with urllib.request.urlopen(request, timeout=600) as response:
            body = json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        raw = exc.read().decode("utf-8", errors="replace") if exc.fp else ""
        details = raw
        try:
            parsed = json.loads(raw) if raw else {}
            details = parsed.get("error") or parsed.get("message") or raw
        except Exception:
            pass

        raise RuntimeError(
            f"Ollama /api/chat failed with HTTP {exc.code}: {details}. "
            "If this mentions context length or memory, reduce NUM_PREDICT or shorten the prompt/model."
        ) from exc
    except urllib.error.URLError as exc:
        raise RuntimeError(
            f"Could not connect to Ollama at {base_url}. "
            "Ensure Ollama is running (`ollama serve`)."
        ) from exc

    message = body.get("message", {})
    content = message.get("content")
    if not content:
        raise RuntimeError(f"Unexpected Ollama response: {body}")
    return content.strip()


In [9]:
transactions = load_transactions(CSV_PATH)
portfolio_summary = summarize_portfolio(transactions)
portfolio_summary


/var/folders/p1/g_l_6ln97jx73n6xnw93ss2h0000gn/T/ipykernel_10332/2916805080.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Activity Date"] = pd.to_datetime(df["Activity Date"], errors="coerce")


{'date_range': {'start': '2020-08-10', 'end': '2026-03-06', 'years': 5.57},
 'transaction_counts': {'rows': 924,
  'buys': 384,
  'sells': 43,
  'distinct_symbols_traded': 61},
 'cash_flows': {'net_deposits': 196787.98,
  'gross_buys': 120684.08,
  'gross_sells': 15979.57,
  'dividends': 1006.44,
  'interest': 3382.45},
 'behavioral_metrics': {'sell_to_buy_ratio': 0.132,
  'annualized_turnover_proxy': 0.024,
  'holding_period_buckets': {'31-364d': 31, '<=30d': 2, '>=365d': 45},
  'median_holding_period_days': 690.0},
 'current_positions_top_15': [{'ticker': 'VOO',
   'quantity': 27.210083,
   'cost_basis_total': 14052.82,
   'portfolio_cost_weight': 13.16},
  {'ticker': 'NVDA',
   'quantity': 140.548243,
   'cost_basis_total': 9882.96,
   'portfolio_cost_weight': 9.25},
  {'ticker': 'AAPL',
   'quantity': 44.1875,
   'cost_basis_total': 9119.43,
   'portfolio_cost_weight': 8.54},
  {'ticker': 'MSFT',
   'quantity': 21.414492,
   'cost_basis_total': 8730.21,
   'portfolio_cost_weight': 

In [10]:
messages = build_messages(
    portfolio_summary=portfolio_summary,
    risk_profile=RISK_PROFILE,
    monthly_budget=MONTHLY_BUDGET,
    sectors=SECTORS,
)

analysis = call_ollama(
    model=MODEL_NAME,
    messages=messages,
    temperature=TEMPERATURE,
    num_predict=NUM_PREDICT,
)

print(analysis)


HTTPError: HTTP Error 500: Internal Server Error